# ***PDF to RAG pipeline***

#### ***Notebook Sections (Cell Mapping)***
1. **Imports & Environment**
    - `Package initialization`
    - `configurations`
    - `Standard data/ML libraries`
    - `Logger configuration`
    - Validation: Confirm CONFIG_PATH exists and config is valid YAML dict 
2. **Utilities Functions  & Text Pre-Processing**
    - `normalize_whitespace`
    - `remove_control_chars`
    - `dedupe_lines`
    - `format_llm_output`
3. **PDF Extraction & Text Cleaning** 
    - `extract_text`
    - `clean_text`
    - `remove_title_section`
    - `remove_ending_section`
    - `save_clean_text`
4. **Tokenization & Chunking**
    - `tokens_of`
    - `detokenize`
    - `extract_key_terms` 
    - `chunk_text`
    - `refine_chunks` `(cl100k_base, 125 tok, 0.20 overlap, 50-700 refined)`
5. **Embeddings & FAISS Index Construction**
    - `build_embeddings_and_index` `(SentenceTransformer all-MiniLM-L6-v2)`
    - `cosine similarity via inner-product`
    - `index persistence` 
6. **Embedding Statistics & Analysis**
    - `Visual: Interactive Embedding Space Projection`
    - `calculate_l2_stats`
    - `sample_pairwise_l2_distances`
    - `calculate_cosine_similarity_stats`
7. **Domain Ontology & Knowledge Graph Construction**
    - `DomainOntology class`
       - ` _load_expanded_vocabulary`
       - `extract_entities_from_text`
    - `KnowledgeGraph class`
       - `add_chunk_to_graph`
       - ` _extract_keywords`
       - `find_related_chunks` 
8. **Semantic Layer & Domain Reasoning**
    - `SemanticLayer class`
       - ` ground_query` 
         - `entity type detection` 
         - `intent classification (synthesis / enumeration / explanatory)`
9.  **Hybrid Retrieval Orchestration**
    - `retrieve_hybrid` (Vector + Graph + Keyword with alpha weighting), Multi-modal fusion logic
    - `synthesize_answer_from_chunks`  (ranked result assembly)
10. **Retriever & Index Loading**
    - `retrieve`
    - `assemble_context`  
11. **LLM Setup & Prompting**
     - `LLMWrapper class`
     - `llm = LLMWrapper` (initializer)
     - `call_llm` (t5-base, prompt engineering with context constraint 374 tokens, citation formatting)
12.  **Enhanced RAG Answer Generation**
     - `enhanced_rag_answer` (semantic grounding → hybrid retrieval → generation → fallback synthesis) (Cross_Encoder)
13.  **Post-Processing, Citation Extraction & Cleanup**
     - `extract_and_remove_citations`  ( Parses inline [chunk_id] markers, deduplicates, returns clean text + indices)
     - `post_process_enhanced` (remove inline citations, fix proper nouns, normalize whitespace, extract citation indices also handles semantic_grounding extraction)
14. **MAIN EXECUTION: Knowledge Graph Initialization & Population**
     - `Initialize: DomainOntology(), KnowledgeGraph(), SemanticLayer()`
     - `Populate graph via kg.add_chunk_to_graph()`
15.  **Core Thesis Query Execution**
     - Execute Query 1 with semantic grounding output, save to `article_answer.txt` 
16.  **Targeted Follow-Up Questions**
     - Generate + answer 5 domain questions iteratively across metrics, geography, barriers, strategy, hybrid retrieval per question, save to `article_qa_pairs.txt`
17.  **Comprehensive Summary Generation**
     - Synthesize cross-chunk insights, enhanced with fallback synthesis logic, save to `article_summary.txt`
19.  **Test**
     - len(final_answer) > 40 characters 
     - leader' in final_answer.lower() 
20.  **Analysis Report & Knowledge Graph Export**
     - Generate JSON report (entities, relationships, retrieval stats), export `knowledge_graph_entities.txt`
21.  **Pipeline Summary & Final Validation**
     - Display entity count, chunks processed, retrieval method, output format, question count, summary length
     - Test Validation Checks:
         * Structural: Verify Q&A count (5/5) and summary sentence range (5-8)
         * Grounding: Confirm all citations reference real source chunks (prevents hallucination)
         * Persistence: Validate all 5 output files exist and are non-empty
         * Data Integrity: Verify JSON report is valid object structure
     - Output: Detailed pass/fail report with verification breakdown
     - **Test Call:** `from test_integration import run_validation; run_validation(qa_results, summary_text, chunks, output_dir)`


`NOTE:`Integration Test Suite**
        - Run `test_integration.py` against the generated artifacts
        - Validates end-to-end pipeline correctness
        - (Can be run separately or embedded as final cell) 

### ***Import & Environment*** 

In [2]:
# Standard Library 
import json
import os
import re
import sys
import textwrap
import time
import traceback
import warnings
from datetime import datetime
import pathlib
from pathlib import Path
from typing import cast, Any, Optional 

# Data Handling & Scientific Computing 
import numpy as np
import pandas as pd
from numpy.typing import NDArray
from sklearn.metrics.pairwise import cosine_similarity

# PDF Processing 
import pdfplumber

# Deep Learning & LLM Frameworks
import tensorflow as tf
import tiktoken
import torch
import yaml
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer, AutoTokenizer, AutoModelForSeq2SeqLM

# Vector Store & Similarity Search 
import faiss
from faiss import IndexFlatL2

# Visualization 
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# Jupyter Configuration 
%matplotlib inline

# System Configuration & Warning Suppression
# Suppress TensorFlow and Deprecation warnings for cleaner notebook output  
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # 0=all, 1=INFO, 2=WARNING, 3=ERROR
warnings.filterwarnings('ignore', message='.*sparse_softmax_cross_entropy.*')
warnings.filterwarnings("ignore", category=DeprecationWarning)
tf.get_logger().setLevel("ERROR")

In [ ]:
# Resolve parent directory to allow importing from local project modules (e.g., logs.logger)
sys.path.append(str(Path("..").resolve()))

# Local Project Imports 
from logs.logger import get_logger

# Initialize centralized logging for pipeline traceability
log = get_logger()
log.info("Pipeline started")

# Define project root and configuration path for relative resource access
PROJECT_ROOT = pathlib.Path("..").resolve()
CONFIG_PATH = pathlib.Path("../configs/config.yaml").resolve()   

# Ensure configuration file exists before attempting to load
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Configuration file not found: {CONFIG_PATH}")

# Load YAML configuration and validate that the root structure is a dictionary
_cfg_raw = yaml.safe_load(CONFIG_PATH.read_text())
if not isinstance(_cfg_raw, dict):
    raise TypeError(
        f"The config file must contain a YAML mapping (dictionary). Got: {type(_cfg_raw)}"
    )
_cfg = _cfg_raw   

# Text segmentation hyperparameters for optimized context windows
CHUNK_SIZE    = _cfg["chunking"]["size"]          
OVERLAP_RATIO = _cfg["chunking"]["overlap"]       

# Hybrid retrieval constraints for balancing vector and graph recall
TOP_K = _cfg["retrieval"]["top_k"]
SCORE_THRESHOLD = _cfg["retrieval"]["score_threshold"]

# Generation settings for the local T5 model to control output length and creativity
MAX_NEW_TOKENS = _cfg["llm"]["max_new_tokens"]
TEMPERATURE    = _cfg["llm"]["temperature"]
SYSTEM_PROMPT  = _cfg["llm"]["system_prompt"]

In [ ]:
print("TOP_K =", TOP_K)
print("LLM will generate up to", MAX_NEW_TOKENS, "tokens")

### ***Utilities Functions  & Text Pre-Processing***

In [ ]:
# UTILITY FUNCTIONS     
def normalize_whitespace(s: str) -> str:
    """
    Normalize various whitespace patterns in text.

    Performs the following transformations:
    - Converts CRLF and CR line endings to LF
    - Collapses consecutive spaces and tabs to single space
    - Reduces 3+ consecutive newlines to exactly 2 newlines
    - Strips leading and trailing whitespace

    Args:
        s (str): Input text to normalize.

    Returns:
        str: Normalized text, or original input if an error occurs.
    """
    try:
        s = re.sub(r'\r\n?', '\n', s)  # Normalize line endings to LF
        s = re.sub(r'[ \t]+', ' ', s)  # Collapse whitespace runs
        s = re.sub(r'\n{3,}', '\n\n', s) # Limit consecutive newlines
        return s.strip()
    except Exception as e:
        print(f"Error in normalize_whitespace: {e}")
        return s  

def remove_control_chars(s: str) -> str:   
    """
    Remove C0 control characters while preserving text content.

    Removes all ASCII control characters (0x00–0x1F, 0x7F) except newline (0x0A/LF),
    which is retained to preserve document structure.

    Args:
        s (str): Input text.

    Returns:
        str: Text with control characters removed, or original input if an error occurs.
    """ 
    try:
        return re.sub(r'[\x00-\x09\x0B-\x1F\x7F]', '', s)
    except Exception as e:
        print(f"Error in remove_control_chars: {e}")
        return s  

def dedupe_lines(s: str) -> str:           
    """
    Remove duplicate lines while preserving order of first occurrence.

    Skips empty or whitespace-only lines entirely and removes any subsequent
    occurrence of a line already seen.

    Args:
        s (str): Input text with newline-separated lines.

    Returns:
        str: Deduplicated text, or original input if an error occurs.
    """
    try:
        lines = []
        seen = set()
        for ln in s.splitlines():
            t = ln.strip()
            # Skip empty lines and skip duplicates (track with set for O(1) lookup)
            if not t:
                continue
            if t in seen: 
                continue
            seen.add(t); lines.append(t)
        return '\n'.join(lines)
    except Exception as e:
        print(f"Error in dedupe_lines: {e}")
        return s
    
def format_llm_output(text: str) -> str:
    """
    Fixes common LLM formatting errors: 
    - Removes spaces before punctuation
    - Ensures sentences start with capital letters
    """
    if not text:
        return ""
    
    # Remove spaces before punctuation (e.g., "landscape ." -> "landscape.")
    text = re.sub(r'\s+([\.!\?,;:])', r'\1', text)
    
    # Fix capitalization: Capitalize the first letter of every sentence
    text = re.sub(r'(^|[.!?]\s+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), text)
    
    # Final whitespace cleanup
    text = ' '.join(text.split())
    
    return text  

### ***PDF Extraction & Text Cleaning***

In [ ]:
def extract_text(pdf_path: str) -> str:    
    """
    Extract text from all pages of a PDF and wrap each with page markers.

    Args:
        pdf_path (str): Path to the PDF file.

    Returns:
        str: Concatenated text from all pages with [PAGE_N_START] and [PAGE_N_END] markers,
             or the original pdf_path string if extraction fails.
    """ 
    try:
        full = []
        with pdfplumber.open(pdf_path) as pdf:
            # Extract text from each page and wrap with identifiable markers for downstream processing
            for i, page in enumerate(pdf.pages, start=1):
                text = page.extract_text() or ""
                if text:
                    full.append(f"[PAGE_{i}_START]\n" + text + f"\n[PAGE_{i}_END]") 
        return "\n\n".join(full)
    except Exception as e:
        print(f"Error in extract_text for {pdf_path}: {e}") 
        return pdf_path  

In [ ]:
def clean_text(raw: str) -> str:     
    """
    Comprehensively clean and normalize extracted text.

    Applies a sequence of transformations: removes control characters, normalizes
    whitespace, deduplicates lines, fixes line-break artifacts, and applies
    heuristics for sentence and word boundary detection.

    Args:
        raw (str): Raw extracted text from PDF or other source.

    Returns:
        str: Cleaned and normalized text, or original input if an error occurs.
    """ 
    try:
        s = remove_control_chars(raw)
        s = normalize_whitespace(s)
        s = dedupe_lines(s)
        
         # Remove hyphens at line breaks to rejoin hyphenated words across lines
        s = s.replace("-\n", "") 
        # Convert lines with excess trailing whitespace to paragraph breaks 
        s = re.sub(r'[ \t]{2,}\n', '\n\n', s) 
        # Join single newlines with spaces (treat as line-wrap continuation, not paragraph break) 
        s = re.sub(r'(?<!\n)\n(?!\n)', ' ', s)
        # Collapse multiple spaces into single space  
        s = re.sub(r' {2,}', ' ', s)
        # Add space between lowercase/digit and uppercase letter (e.g., "exampleText" → "example Text")  
        s = re.sub(r'(?<=[a-z0-9\)\]\-,:;])(?=[A-Z])', ' ', s)
        # Collapse multiple newlines into exactly two  
        s = re.sub(r'\n{3,}', '\n\n', s)
        # Remove newlines from very long lines (likely OCR artifacts from wrapped text) 
        s = re.sub(r'([^\n]{1000,}?)\n', lambda m: m.group(1) + ' ', s)
        # Placeholder for potential future merged-word handling (currently no-op)
        s = re.sub(r'(?<=\w)(?=\w)', '', s)  
        return s.strip()
    except Exception as e:
        print(f"Error in clean_text: {e}")
        return raw 

In [ ]:
def remove_title_section(text: str, start_marker: str = "In the 21st century") -> str:    
    """
    Trim text to start from a specified marker, discarding all preceding content.

    Args:
        text (str): Input text.
        start_marker (str): The text marker from which to begin output. Default is "In the 21st century".

    Returns:
        str: Text from the marker onward (stripped), or original text unchanged if marker is not found.
    """ 
    try:
        if start_marker in text:
            idx = text.find(start_marker)
            return text[idx:].strip()
        return text
    except Exception as e:
        print(f"Error in remove_title_section: {e}")
        return text
    
def remove_ending_section(text: str, end_marker: str = "Gen. article Gemini Flash 2.0") -> str:
    """
    Remove text from a specified marker to the end, keeping only the preceding content.

    Args:
        text (str): Input text.
        end_marker (str): The marker from which to remove content onward. Default is the source attribution.

    Returns:
        str: Text up to the marker (stripped), or original text unchanged if marker is not found.
    """
    try:
        if end_marker in text:
            idx = text.find(end_marker)
            return text[:idx].strip()
        return text
    except Exception as e:
        print(f"Error in remove_ending_section: {e}")
        return text

In [ ]:
# Load PDF, extract and clean text, remove leading title section, preview, and save to file   
pdf_path = PROJECT_ROOT / "data" / "Tech_article.pdf"

raw = extract_text(str(pdf_path))
clean = clean_text(raw)
clean = remove_title_section(clean, "In the 21st century")  # Remove title
clean = remove_ending_section(clean, " Gen. article Gemini Flash 2.0 [PAGE_2_END]") # Remove ending
print("Clean 1st sentence preview:\n", clean[:283])  
print("Clean last sentence preview:\n", clean[-293:]) 

output_path = PROJECT_ROOT / "data" 

with open(output_path / "tech_article.txt", "w", encoding="utf8") as f:
    f.write(clean)

### ***Tokenization & Chunking***

In [ ]:
ENC = tiktoken.get_encoding("cl100k_base")     

def tokens_of(text: str) -> list[int]:
    """
    Encode text into token IDs using the configured tokenizer.
    
    Args:
        text: The input text string to encode.
        
    Returns:
        A list of integer token IDs. Returns an empty list if encoding fails.
    """ 
    try:
        return ENC.encode(text)
    except Exception as e:
        print(f"Error in tokens_of function: {e}")
        return []

def detokenize(tokens: list[int]) -> str:
    """
    Decode token IDs back into text using the configured tokenizer.
    
    Args:
        tokens: A list of integer token IDs to decode.
        
    Returns:
        The decoded text string. Returns an empty string if decoding fails.
    """
    try:
        return ENC.decode(tokens)
    except Exception as e:
        print(f"Error in detokenize function: {e}")
        return ""

In [ ]:
def extract_key_terms(query: str) -> list[str]:        
    """
    Extract meaningful keywords from a query by removing common stop words.
    
    Args:
        query: The input query string to process.
        
    Returns:
        A list of filtered keywords (non-stop-words with length > 2 characters).
    """ 
    stop_words = {
        'what', 'is', 'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 
        'to', 'for', 'of', 'with', 'by', 'from', 'as', 'are', 'was', 'be', 
        'have', 'has', 'do', 'does', 'did', 'will', 'can', 'could', 'should',
        'help', 'address', 'why', 'how', 'who', 'where', 'when'
    }
    
    # Convert to lowercase, split into words, remove punctuation, and filter stop words
    words = query.lower().split()
    keywords = [w.strip('?,.!;:') for w in words if w.lower() not in stop_words and len(w) > 2]
    
    return keywords

In [ ]:
def chunk_text(text: str, chunk_size_tokens: int = CHUNK_SIZE, overlap_ratio: float = OVERLAP_RATIO) -> list[dict]:     
    """
    Split text into token-limited chunks while preserving paragraph and sentence boundaries.

    Chunks are kept near the requested token size by respecting paragraph breaks first,
    then sentence boundaries. Overlap parameter is reserved for future use.

    Args:
        text: The input text to chunk.
        chunk_size_tokens: Target maximum token count per chunk (default: 400).
        overlap_ratio: Placeholder parameter for future overlap implementation (default: 0.2).

    Returns:
        A list of dictionaries, each containing chunk metadata:
        - id: Unique chunk identifier
        - text: The chunk text content
        - token_count: Number of tokens in the chunk
        - token_start/token_end: Token position markers (simplified; reserved for future use)
        - source: Source filename
        - chunk_index: Sequential chunk index
    """ 
    try:
        # Split text into paragraphs to keep related content together and prevent mixing different topics
        paragraphs = re.split(r'\n\s*\n', text)
        chunks = []
        idx = 0
        
        for para in paragraphs:
            para = para.strip()
            if not para: 
                continue
            
            # Count tokens in the paragraph to determine chunking strategy
            para_tokens = tokens_of(para)
            
            if len(para_tokens) <= chunk_size_tokens:
                # Paragraph is within limits; preserve it as a single coherent chunk
                chunks.append({
                    "id": f"tech_article_chunk_{idx}",
                    "text": para,
                    "token_count": len(para_tokens),
                    "token_start": 0, 
                    "token_end": len(para_tokens),
                    "source": os.path.basename(pdf_path),
                    "chunk_index": idx
                })
                idx += 1
            else:
                # Paragraph exceeds token limit; split into sentences for finer control
                sentences = re.split(r'(?<=[.!?])\s+', para)   # Split after sentence-ending punctuation
                current_chunk_text = ""
                
                for sent in sentences:
                    # Accumulate sentences until adding the next one would exceed the token limit
                    combined_text = (current_chunk_text + " " + sent).strip()
                    if len(tokens_of(combined_text)) > chunk_size_tokens:
                        # Save accumulated sentences as a chunk before exceeding the limit
                        if current_chunk_text:
                            chunks.append({
                                "id": f"tech_article_chunk_{idx}",
                                "text": current_chunk_text.strip(),
                                "token_count": len(tokens_of(current_chunk_text)),
                                "token_start": 0,
                                "token_end": len(tokens_of(current_chunk_text)),
                                "source": os.path.basename(pdf_path),
                                "chunk_index": idx
                            })
                            idx += 1
                            current_chunk_text = sent
                        else:
                            # Edge case: single sentence exceeds token limit; store as-is
                            current_chunk_text = sent 
                    else:
                        # Sentence fits; add it to the current accumulating chunk
                        current_chunk_text += " " + sent
                
                # Finalize the last accumulated chunk for this paragraph
                if current_chunk_text:
                    chunks.append({
                        "id": f"tech_article_chunk_{idx}",
                        "text": current_chunk_text.strip(),
                        "token_count": len(tokens_of(current_chunk_text)),
                        "token_start": 0,
                        "token_end": len(tokens_of(current_chunk_text)),
                        "source": os.path.basename(pdf_path),
                        "chunk_index": idx
                    })
                    idx += 1
                    
        return chunks
    except Exception as e:
        print(f"Error in chunk_text function: {e}")
        return []

try:
    # Apply chunking strategy with 125-token limit for optimal embedding performance and retrieval
    chunks = chunk_text(clean)      # (clean, chunk_size_tokens=125,overlap_ratio=0.20)          
    print(f"Produced {len(chunks)} chunks. Token counts:", [c["token_count"] for c in chunks[:8]])
    
    # Persist chunks to JSON file for downstream processing and evaluation
    with open(output_path / "chunks.json", "w", encoding="utf8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    
except Exception as e:
    print(f"General error: {e}")

In [ ]:
def refine_chunks(chunks: list[dict], min_tokens=50, max_tokens=700) -> list[dict]:        
    """
    Adjust chunk sizes by merging under-sized chunks and splitting over-sized ones.
    
    Ensures consistent chunk quality by merging fragments below min_tokens into 
    previous chunks and splitting chunks exceeding max_tokens at paragraph boundaries.
    Reindexes all chunks sequentially after refinement.

    Args:
        chunks: List of chunk dictionaries containing 'text', 'token_count', and other metadata.
        min_tokens: Minimum acceptable token count; smaller chunks are merged (default: 50).
        max_tokens: Maximum acceptable token count; larger chunks are split (default: 700).

    Returns:
        A list of refined and reindexed chunk dictionaries with updated token counts, 
        ids, and chunk_index values.
    """
    out = []
    try:
        for c in chunks:
            if c["token_count"] < min_tokens and out:
                # Merge small chunk into previous chunk to avoid fragments that hurt retrieval
                out[-1]["text"] += "\n\n" + c["text"]
                out[-1]["token_end"] = c["token_end"]
                out[-1]["token_count"] = len(tokens_of(out[-1]["text"]))
            elif c["token_count"] > max_tokens:
                # Split oversized chunks at paragraph boundaries to maintain semantic coherence
                text = c["text"]
                parts = text.split("\n\n")
                if len(parts) > 1:
                    cur = ""
                    for p in parts:
                        # Attempt to accumulate paragraphs without exceeding max token limit
                        cand = (cur + "\n\n" + p).strip() if cur else p
                        if len(tokens_of(cand)) > max_tokens:
                            # Current accumulation is full; start a new chunk with this paragraph
                            if cur:
                                out.append({**c, "text": cur})
                            cur = p
                        else:
                            # Paragraph fits; add it to the current accumulation
                            cur = cand
                            # Save the final accumulated chunk for this oversized section
                    if cur:
                        out.append({**c, "text": cur})
                else:
                    # Fallback: no paragraph breaks found; split text at character midpoint
                    mid = len(text) // 2
                    a = text[:mid]
                    b = text[mid:]
                    out.append({**c, "text": a})
                    out.append({**c, "text": b})
            else:
                # Chunk size is within acceptable range; keep as-is
                out.append(c)
        
        # Re-assign sequential identifiers and recount tokens after merges and splits
        for i, c in enumerate(out):
            c["id"] = f"Tech_article_chunk_{i}"
            c["chunk_index"] = i
            c["token_count"] = len(tokens_of(c["text"]))
        return out
    except Exception as e:
        print(f"Error in refine_chunks function: {e}")
        return []

try:
    # Apply size constraints to ensure consistent chunk quality and optimal retrieval performance
    chunks = refine_chunks(chunks)
    print("Refined chunks:", len(chunks))
    
    # Persist refined chunks to disk for downstream embedding and indexing
    with open(output_path / "chunks_refined.json", "w", encoding="utf8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

except Exception as e:
    print(f"General error: {e}")

### ***Embeddings & FAISS Index Construction***

In [ ]:
# Vector store initialization
EMB_MODEL = "all-MiniLM-L6-v2"  # Compact embedding model optimized for semantic search with low latency    
embedder = SentenceTransformer(EMB_MODEL) # Initialize the model once to avoid redundant loading
